# New Bird Onboarding Pipeline (Temporary + Approval-based)

This notebook automates onboarding a new bird (example: **Emperor penguin**) in safe stages:

1. Fetch reliable species metadata (Wikipedia/Wikidata first)
2. Ask Gemini to map/create group (tier 1/2) + group description
3. Fetch candidate images from Wikimedia Commons
4. Validate/filter images and keep top 5
5. Build temporary payload
6. Integrate into app data **only after explicit approval**

All intermediate data is saved under `tmp/new_bird_onboarding/<bird_slug>/`.

In [1]:
# Block 1: Setup, config, and shared helpers
import os
import re
import json
import time
import shutil
import hashlib
from pathlib import Path
from io import BytesIO

import requests
from PIL import Image

# ==== User inputs ====
TARGET_BIRD = "Emperor penguin"
USER_AGENT = "BirdWhoBot/3.0 (educational app; onboarding pipeline)"

# Optional: set directly or export as env var GEMINI_API_KEY
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")

# Safety switch for final write
APPLY_CHANGES = False

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(6):
        if (cur / "data" / "quiz_tiers_data.json").exists() and (cur / "assets").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise FileNotFoundError("Could not locate repo root (missing data/quiz_tiers_data.json).")

REPO_ROOT = find_repo_root(Path.cwd())
DATA_DIR = REPO_ROOT / "data"
ASSETS_DIR = REPO_ROOT / "assets"
BIRDS_ASSETS_DIR = ASSETS_DIR / "birds"

def slugify(name: str) -> str:
    s = name.strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    return re.sub(r"_+", "_", s).strip("_")

BIRD_SLUG = slugify(TARGET_BIRD)
TMP_ROOT = REPO_ROOT / "tmp" / "new_bird_onboarding" / BIRD_SLUG
TMP_RAW_IMAGES = TMP_ROOT / "raw_images"
TMP_SELECTED_IMAGES = TMP_ROOT / "selected_images"
TMP_ROOT.mkdir(parents=True, exist_ok=True)
TMP_RAW_IMAGES.mkdir(parents=True, exist_ok=True)
TMP_SELECTED_IMAGES.mkdir(parents=True, exist_ok=True)

def save_json(path: Path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})

print("✅ Setup complete")
print(f"Repo root: {REPO_ROOT}")
print(f"Target bird: {TARGET_BIRD} (slug={BIRD_SLUG})")
print(f"Temp folder: {TMP_ROOT}")
print(f"Gemini key provided: {'yes' if GEMINI_API_KEY else 'no'}")

✅ Setup complete
Repo root: C:\Users\matan\Documents\BirdWho_ag
Target bird: Emperor penguin (slug=emperor_penguin)
Temp folder: C:\Users\matan\Documents\BirdWho_ag\tmp\new_bird_onboarding\emperor_penguin
Gemini key provided: yes


In [ ]:
# Block 2: Fetch reliable metadata from Wikipedia + Wikidata
# Also creates SHORT description in en/he/es/fr (Gemini preferred)

import importlib

WIKI_API = "https://en.wikipedia.org/w/api.php"
WIKIDATA_API = "https://www.wikidata.org/w/api.php"


def _import_genai_optional():
    try:
        from google import genai
        return genai
    except Exception:
        pass
    try:
        return importlib.import_module("google.genai")
    except Exception:
        return None


def wiki_get_page(title: str):
    params = {
        "action": "query",
        "format": "json",
        "prop": "extracts|langlinks|pageprops|categories",
        "exintro": 1,
        "explaintext": 1,
        "redirects": 1,
        "titles": title,
        "lllimit": 500,
        "cllimit": 200,
    }
    r = session.get(WIKI_API, params=params, timeout=30)
    r.raise_for_status()
    pages = r.json().get("query", {}).get("pages", {})
    page = next(iter(pages.values()))
    if "missing" in page:
        raise ValueError(f"Wikipedia page not found: {title}")
    return page


def wikidata_entity(entity_id: str):
    params = {
        "action": "wbgetentities",
        "format": "json",
        "ids": entity_id,
        "props": "labels|descriptions|claims",
        "languages": "en|he|es|fr",
    }
    r = session.get(WIKIDATA_API, params=params, timeout=30)
    r.raise_for_status()
    entities = r.json().get("entities", {})
    return entities.get(entity_id, {})


def extract_scientific_name(entity: dict) -> str:
    claims = entity.get("claims", {})
    p225 = claims.get("P225", [])
    if not p225:
        return ""
    try:
        return p225[0]["mainsnak"]["datavalue"]["value"]
    except Exception:
        return ""


def extract_lang_name(page: dict, lang: str, fallback: str = "") -> str:
    for item in page.get("langlinks", []):
        if item.get("lang") == lang:
            return item.get("*", fallback)
    return fallback


def infer_category(text_en: str, categories: list[str]) -> str:
    t = (text_en or "").lower()
    c = " ".join(categories).lower()
    if any(k in t or k in c for k in ["penguin", "seabird", "marine", "ocean"]):
        return "Water Bird"
    if any(k in t or k in c for k in ["owl", "eagle", "falcon", "hawk", "vulture"]):
        return "Raptor"
    if any(k in t or k in c for k in ["duck", "goose", "swan", "heron", "ibis", "stork"]):
        return "Water Bird"
    return "Songbird"


def _first_sentence(text: str) -> str:
    text = (text or "").strip()
    if not text:
        return ""
    parts = re.split(r"(?<=[.!?])\s+", text)
    s = parts[0].strip() if parts else text
    return s[:240].strip()


def summarize_multilingual_descriptions(names: dict, scientific_name: str, wiki_extract: str, wikidata_desc: dict):
    # Fallback first
    fallback = {
        "en": _first_sentence(wiki_extract) or f"{names.get('en', TARGET_BIRD)} is a bird species.",
        "he": (wikidata_desc.get("he", {}).get("value", "") or "מין של עוף")[:220],
        "es": (wikidata_desc.get("es", {}).get("value", "") or "especie de ave")[:220],
        "fr": (wikidata_desc.get("fr", {}).get("value", "") or "espèce d'oiseau")[:220],
    }

    genai = _import_genai_optional()
    if not GEMINI_API_KEY or genai is None:
        return fallback, "fallback-no-gemini"

    try:
        client = genai.Client(api_key=GEMINI_API_KEY)
        prompt = f"""
You are writing short educational bird descriptions.

Bird:
- English name: {names.get('en', '')}
- Scientific name: {scientific_name}

Source summary from Wikipedia (must be the factual basis):
{wiki_extract[:1000]}

Task:
Return ONLY valid JSON object with keys: en, he, es, fr
Rules:
- Each value: 1-2 short sentences, max 220 characters.
- Keep scientific facts accurate to source.
- No markdown. No extra keys.
"""
        resp = client.models.generate_content(model="gemini-2.5-flash", contents=prompt)
        txt = (resp.text or "").strip()
        if txt.startswith("```json"):
            txt = txt[7:]
        if txt.endswith("```"):
            txt = txt[:-3]
        obj = json.loads(txt.strip())

        out = {
            "en": str(obj.get("en", fallback["en"]))[:220],
            "he": str(obj.get("he", fallback["he"]))[:220],
            "es": str(obj.get("es", fallback["es"]))[:220],
            "fr": str(obj.get("fr", fallback["fr"]))[:220],
        }
        return out, "gemini"
    except Exception:
        return fallback, "fallback-gemini-error"


page = wiki_get_page(TARGET_BIRD)
wikidata_id = page.get("pageprops", {}).get("wikibase_item", "")
entity = wikidata_entity(wikidata_id) if wikidata_id else {}

extract_en = (page.get("extract") or "").strip()
categories = [x.get("title", "") for x in page.get("categories", [])]

names = {
    "en": page.get("title", TARGET_BIRD),
    "he": extract_lang_name(page, "he", page.get("title", TARGET_BIRD)),
    "es": extract_lang_name(page, "es", page.get("title", TARGET_BIRD)),
    "fr": extract_lang_name(page, "fr", page.get("title", TARGET_BIRD)),
}

scientific_name = extract_scientific_name(entity) or "Unknown"
desc, desc_source = summarize_multilingual_descriptions(names, scientific_name, extract_en, entity.get("descriptions", {}))

bird_meta = {
    "name": BIRD_SLUG,
    "wikiTitle": page.get("title", TARGET_BIRD),
    "scientificName": scientific_name,
    "names": names,
    "description": desc,
    "category": infer_category(extract_en, categories),
    "difficulty": 3,
    "entryType": "species",
    "tier": 3,
    "sources": {
        "wikipedia": f"https://en.wikipedia.org/wiki/{page.get('title', TARGET_BIRD).replace(' ', '_')}",
        "wikidata": f"https://www.wikidata.org/wiki/{wikidata_id}" if wikidata_id else "",
        "description_builder": desc_source,
    },
}

save_json(TMP_ROOT / "meta_wiki.json", bird_meta)

print("✅ Metadata fetched")
print(json.dumps({
    "name": bird_meta["name"],
    "scientificName": bird_meta["scientificName"],
    "names": bird_meta["names"],
    "category": bird_meta["category"],
    "description": {k: v for k, v in bird_meta["description"].items()},
    "description_source": bird_meta["sources"]["description_builder"],
    "source": bird_meta["sources"]["wikipedia"],
}, ensure_ascii=False, indent=2))

✅ Metadata fetched
{
  "name": "emperor_penguin",
  "scientificName": "Aptenodytes forsteri",
  "names": {
    "en": "Emperor penguin",
    "he": "פינגווין קיסרי",
    "es": "Aptenodytes forsteri",
    "fr": "Manchot empereur"
  },
  "category": "Water Bird",
  "description": {
    "en": "The Emperor penguin (Aptenodytes forsteri) is the tallest and heaviest of all penguins, endemic to Antarctica. This flightless marine bird has striking black, white, and yellow plumage, and dives deep for fish and krill.",
    "he": "הפינגווין הקיסרי (Aptenodytes forsteri) הוא הגדול והכבד מכל הפינגווינים החיים, ואנדמי לאנטארקטיקה. הוא בעל נוצות שחורות, לבנות וצהובות, צולל לעומקים בחיפוש אחר דגים וסרטנים זעירים.",
    "es": "El pingüino emperador (Aptenodytes forsteri) es el más alto y pesado de todos los pingüinos, endémico de la Antártida. Esta ave marina no voladora, con plumaje negro, blanco y amarillo, bucea profundo por peces y krill.",
    "fr": "Le manchot empereur (Aptenodytes forsteri) est le

In [ ]:
# Block 3: Group detection/creation using Gemini API (tier 1 or 2)
# Requires: pip install google-genai

import sys
import importlib

qt_data = load_json(DATA_DIR / "quiz_tiers_data.json")
entities = qt_data.get("entities", {})

existing_groups = {}
for entity_id, ent in entities.items():
    if ent.get("entryType") == "group":
        gid = str(ent.get("groupId") or ent.get("name", "")).strip()
        if not gid:
            continue
        existing_groups[gid] = {
            "entityId": entity_id,
            "groupId": gid,
            "tier": ent.get("tier", 2),
            "names": ent.get("names", {}),
        }


def _import_genai():
    try:
        from google import genai
        return genai
    except Exception:
        pass

    try:
        module = importlib.import_module("google.genai")
        return module
    except Exception as e:
        raise ImportError(
            "Cannot import google-genai in this notebook kernel. "
            f"Install in this exact kernel with: {sys.executable} -m pip install -U google-genai"
        ) from e


def _normalize_group_id(value: str, fallback_en: str) -> str:
    # Keep app-consistent IDs: lowercase snake_case
    raw = (value or "").strip().lower()
    raw = re.sub(r"[^a-z0-9]+", "_", raw)
    raw = re.sub(r"_+", "_", raw).strip("_")
    if raw:
        return raw

    fb = (fallback_en or "group").strip().lower()
    fb = re.sub(r"[^a-z0-9]+", "_", fb)
    fb = re.sub(r"_+", "_", fb).strip("_")
    return fb or "group"


def decide_group_with_gemini(meta: dict, groups: dict, api_key: str):
    if not api_key:
        raise RuntimeError("GEMINI_API_KEY is missing. Set it and re-run this block.")

    genai = _import_genai()
    client = genai.Client(api_key=api_key)

    compact_groups = [
        {
            "groupId": g["groupId"],
            "tier": g.get("tier", 2),
            "name_en": g.get("names", {}).get("en", ""),
            "name_he": g.get("names", {}).get("he", ""),
        }
        for g in groups.values()
    ]

    prompt = f"""
You are classifying a NEW bird species for a quiz app.

Bird metadata:
{json.dumps(meta, ensure_ascii=False, indent=2)}

Existing groups in app:
{json.dumps(compact_groups, ensure_ascii=False, indent=2)}

Task:
1) Decide if this bird should join an existing group.
2) If no suitable group exists, create a new group.
3) Birds of the same general type/family should be grouped together.
4) Group tier must be 1 or 2 only.
5) groupId MUST be lowercase snake_case in English letters/numbers only.
6) Return short group descriptions in en/he/es/fr.

Output ONLY raw JSON object with this exact schema:
{{
  "useExistingGroup": true/false,
  "groupId": "string",
  "groupNames": {{"en":"","he":"","es":"","fr":""}},
  "tier": 1 or 2,
  "description": {{"en":"","he":"","es":"","fr":""}},
  "reason": "short explanation"
}}
"""

    resp = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )
    txt = (resp.text or "").strip()
    if txt.startswith("```json"):
        txt = txt[7:]
    if txt.endswith("```"):
        txt = txt[:-3]

    result = json.loads(txt.strip())
    result["groupId"] = _normalize_group_id(
        str(result.get("groupId", "")),
        str((result.get("groupNames") or {}).get("en", "")),
    )

    # Match existing groups case-insensitively to avoid duplicates
    lower_map = {k.lower(): k for k in groups.keys()}
    if result["groupId"].lower() in lower_map:
        result["groupId"] = lower_map[result["groupId"].lower()]

    if result.get("tier") not in [1, 2]:
        result["tier"] = 2
    return result


group_decision = decide_group_with_gemini(bird_meta, existing_groups, GEMINI_API_KEY)

group_id = group_decision["groupId"]
group_entity_key = f"group_{group_id}"
already_exists = group_id in existing_groups or group_entity_key in entities
group_decision["alreadyExistsInApp"] = bool(already_exists)

save_json(TMP_ROOT / "group_decision.json", group_decision)

print("✅ Group decision ready")
print(json.dumps(group_decision, ensure_ascii=False, indent=2))

✅ Group decision ready
{
  "useExistingGroup": false,
  "groupId": "Penguin",
  "groupNames": {
    "en": "Penguin",
    "he": "פינגווין",
    "es": "Pingüino",
    "fr": "Manchot"
  },
  "tier": 1,
  "description": {
    "en": "Flightless aquatic birds known for their upright stance and specialized wings for swimming in the Southern Hemisphere.",
    "he": "עופות ימיים חסרי יכולת תעופה, המפורסמים בעמידתם הזקופה ובכנפיהם המותאמות לשחייה בחצי הכדור הדרומי.",
    "es": "Aves marinas no voladoras conocidas por su postura erguida y alas especializadas para nadar en el hemisferio sur.",
    "fr": "Oiseaux marins incapables de voler, connus pour leur posture droite et leurs ailes spécialisées pour nager dans l'hémisphère sud."
  },
  "alreadyExistsInApp": false
}


In [6]:
# Block 4: Fetch candidate images from Wikimedia Commons into temporary folder
COMMONS_API = "https://commons.wikimedia.org/w/api.php"

BAD_TITLE_RE = re.compile(
    r"illustration|drawing|sketch|painting|diagram|map|range_map|skeleton|skull|taxidermy|specimen|stamp|logo|coin|eggs?|nest|feather",
    re.IGNORECASE,
)


def commons_search(query: str, limit: int = 30):
    params = {
        "action": "query",
        "format": "json",
        "generator": "search",
        "gsrnamespace": 6,
        "gsrsearch": query,
        "gsrlimit": limit,
        "prop": "imageinfo",
        "iiprop": "url|mime|size",
        "iiurlwidth": 1200,
    }
    r = session.get(COMMONS_API, params=params, timeout=30)
    r.raise_for_status()
    pages = r.json().get("query", {}).get("pages", {})

    out = []
    for p in pages.values():
        for ii in p.get("imageinfo", []):
            mime = ii.get("mime", "")
            if mime not in {"image/jpeg", "image/jpg"}:
                continue
            out.append(
                {
                    "title": p.get("title", ""),
                    "url": ii.get("thumburl") or ii.get("url", ""),
                    "width": ii.get("thumbwidth", ii.get("width", 0)),
                    "height": ii.get("thumbheight", ii.get("height", 0)),
                }
            )
    return out


def gather_candidates(meta: dict):
    en = meta["names"]["en"]
    sci = meta.get("scientificName", "")
    queries = [
        f'"{en}" bird photo',
        f'"{en}" bird wildlife',
        f'"{en}" bird in flight',
        f'"{en}" male female juvenile',
    ]
    if sci and sci != "Unknown":
        queries.insert(0, f'"{sci}"')

    seen = set()
    out = []
    for q in queries:
        try:
            res = commons_search(q, limit=30)
            for item in res:
                u = item.get("url", "")
                if not u or u in seen:
                    continue
                if item["width"] < 450 or item["height"] < 320:
                    continue
                if BAD_TITLE_RE.search(item.get("title", "")):
                    continue
                seen.add(u)
                item["query"] = q
                out.append(item)
            time.sleep(0.2)
        except Exception as ex:
            print(f"search failed for query={q}: {ex}")
    return out


def download_candidate_images(candidates: list[dict], out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    downloaded = []

    for i, c in enumerate(candidates, start=1):
        try:
            r = session.get(c["url"], timeout=30)
            r.raise_for_status()
            img = Image.open(BytesIO(r.content)).convert("RGB")
            fname = f"cand_{i:03d}.jpg"
            fpath = out_dir / fname
            img.save(fpath, "JPEG", quality=92)
            downloaded.append({**c, "file": fname, "path": str(fpath)})
        except Exception:
            continue

    return downloaded


cands = gather_candidates(bird_meta)
downloaded = download_candidate_images(cands, TMP_RAW_IMAGES)
save_json(TMP_ROOT / "image_candidates.json", downloaded)

print("✅ Image fetch complete")
print(json.dumps({
    "candidate_count": len(cands),
    "downloaded_count": len(downloaded),
    "raw_dir": str(TMP_RAW_IMAGES),
}, ensure_ascii=False, indent=2))

✅ Image fetch complete
{
  "candidate_count": 40,
  "downloaded_count": 5,
  "raw_dir": "C:\\Users\\matan\\Documents\\BirdWho_ag\\tmp\\new_bird_onboarding\\emperor_penguin\\raw_images"
}


In [7]:
# Block 5: Validate/filter images and keep top 5 only
# Optional deps: pip install imagehash numpy open-clip-torch torch

try:
    import imagehash
except Exception:
    imagehash = None

CLIP_READY = False
_clip_ctx = {}


def try_load_clip():
    global CLIP_READY
    try:
        import torch
        import open_clip

        dev = "cuda" if torch.cuda.is_available() else "cpu"
        model, _, prep = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")
        tok = open_clip.get_tokenizer("ViT-B-32")
        _clip_ctx.update({"torch": torch, "model": model.to(dev).eval(), "prep": prep, "tok": tok, "dev": dev})
        CLIP_READY = True
    except Exception as ex:
        print(f"CLIP not available: {ex}")
        CLIP_READY = False


def clip_score_image(path: Path, bird_name: str):
    import numpy as np

    torch = _clip_ctx["torch"]
    model = _clip_ctx["model"]
    prep = _clip_ctx["prep"]
    tok = _clip_ctx["tok"]
    dev = _clip_ctx["dev"]

    img = Image.open(path).convert("RGB")
    timg = prep(img).unsqueeze(0).to(dev)

    pos_prompts = [
        f"a wildlife photograph of a {bird_name}",
        f"a photo of a {bird_name} bird outdoors",
        f"a clear full-body photo of a {bird_name}",
    ]
    neg_prompts = [
        "a map or infographic",
        "a bird illustration or drawing",
        "a photo with no visible bird",
        "a close-up of eggs or feathers only",
    ]

    with torch.no_grad():
        img_f = model.encode_image(timg)
        img_f = img_f / img_f.norm(dim=-1, keepdim=True)

        p = tok(pos_prompts).to(dev)
        n = tok(neg_prompts).to(dev)
        pf = model.encode_text(p)
        pf = pf / pf.norm(dim=-1, keepdim=True)
        nf = model.encode_text(n)
        nf = nf / nf.norm(dim=-1, keepdim=True)

        img_np = img_f.squeeze(0).cpu().numpy()
        pos_score = float((pf.cpu().numpy() @ img_np).mean())
        neg_score = float((nf.cpu().numpy() @ img_np).max())

    return pos_score, neg_score


def phash_dedup(paths: list[Path], max_dist=8):
    if imagehash is None:
        return paths
    kept = []
    hashes = []
    for p in paths:
        try:
            h = imagehash.phash(Image.open(p).convert("RGB"))
            if all((h - old) >= max_dist for old in hashes):
                kept.append(p)
                hashes.append(h)
        except Exception:
            continue
    return kept


try_load_clip()

raw_items = load_json(TMP_ROOT / "image_candidates.json")
raw_paths = [Path(x["path"]) for x in raw_items if Path(x["path"]).exists()]
dedup_paths = phash_dedup(raw_paths, max_dist=8)

scored = []
for p in dedup_paths:
    try:
        img = Image.open(p)
        w, h = img.size

        if CLIP_READY:
            pos, neg = clip_score_image(p, bird_meta["names"]["en"])
            valid = pos > neg and pos > 0.18
            score = pos - neg
        else:
            pos, neg = (0.0, 0.0)
            valid = w >= 500 and h >= 350
            score = (w * h) / 1_000_000

        if valid:
            scored.append(
                {
                    "path": str(p),
                    "file": p.name,
                    "width": w,
                    "height": h,
                    "clip_pos": round(pos, 4),
                    "clip_neg": round(neg, 4),
                    "score": round(float(score), 5),
                }
            )
    except Exception:
        continue

scored.sort(key=lambda x: x["score"], reverse=True)
top5 = scored[:5]

for i, it in enumerate(top5, start=1):
    src = Path(it["path"])
    dst = TMP_SELECTED_IMAGES / f"{BIRD_SLUG}_{i}.jpg"
    img = Image.open(src).convert("RGB")
    if img.width > 1000 or img.height > 1000:
        img.thumbnail((1000, 1000), Image.Resampling.LANCZOS)
    img.save(dst, "JPEG", quality=82, optimize=True)
    it["selected_file"] = dst.name

save_json(TMP_ROOT / "top5_images.json", top5)

print("✅ Image filtering complete")
print(json.dumps({
    "raw_images": len(raw_paths),
    "after_dedup": len(dedup_paths),
    "valid_scored": len(scored),
    "selected_top5": len(top5),
    "selected_dir": str(TMP_SELECTED_IMAGES),
    "clip_used": CLIP_READY,
}, ensure_ascii=False, indent=2))
if top5:
    print("Top files:", [x["selected_file"] for x in top5])

CLIP not available: No module named 'open_clip'
✅ Image filtering complete
{
  "raw_images": 5,
  "after_dedup": 5,
  "valid_scored": 5,
  "selected_top5": 5,
  "selected_dir": "C:\\Users\\matan\\Documents\\BirdWho_ag\\tmp\\new_bird_onboarding\\emperor_penguin\\selected_images",
  "clip_used": false
}
Top files: ['emperor_penguin_1.jpg', 'emperor_penguin_2.jpg', 'emperor_penguin_3.jpg', 'emperor_penguin_4.jpg', 'emperor_penguin_5.jpg']


In [16]:
# Block 6: Build temporary payload (species + optional new group)
group_decision = load_json(TMP_ROOT / "group_decision.json")
top5 = load_json(TMP_ROOT / "top5_images.json")

selected_image_files = [x.get("selected_file") for x in top5 if x.get("selected_file")]
if len(selected_image_files) < 5:
    raise RuntimeError("Need 5 validated images before integration. Re-run Block 4-5 with better candidates.")

species_entity = {
    "id": BIRD_SLUG,
    "name": BIRD_SLUG,
    "names": bird_meta["names"],
    "scientificName": bird_meta["scientificName"],
    "difficulty": 3,
    "tier": 3,
    "entryType": "species",
    "category": bird_meta["category"],
    "locations": [],
    "images": selected_image_files,
    "image": BIRD_SLUG,
    "groupId": group_decision["groupId"],
    "isGroup": False,
    "description": bird_meta["description"],
}

new_group_entity = None
if not group_decision.get("alreadyExistsInApp", False):
    gid = group_decision["groupId"]
    new_group_entity = {
        "id": f"group_{gid}",
        "name": gid,
        "names": group_decision["groupNames"],
        "scientificName": "Various",
        "difficulty": int(group_decision["tier"]),
        "tier": int(group_decision["tier"]),
        "entryType": "group",
        "category": bird_meta["category"],
        "locations": [],
        "tags": [],
        "images": selected_image_files[:3],
        "image": BIRD_SLUG,
        "groupId": gid,
        "isGroup": True,
        "description": group_decision["description"],
    }

payload = {
    "birdSlug": BIRD_SLUG,
    "species": species_entity,
    "group": new_group_entity,
    "groupDecision": group_decision,
    "images": selected_image_files,
    "tempPath": str(TMP_ROOT),
    "createdAt": time.strftime("%Y-%m-%d %H:%M:%S"),
}

save_json(TMP_ROOT / "new_bird_payload.json", payload)

print("✅ Temporary payload prepared")
print(json.dumps({
    "bird": payload["species"]["names"]["en"],
    "groupId": payload["species"]["groupId"],
    "group_is_new": payload["group"] is not None,
    "group_description": payload["group"]["description"]["en"] if payload["group"] else "",
    "images": payload["images"],
    "payload_path": str(TMP_ROOT / "new_bird_payload.json"),
}, ensure_ascii=False, indent=2))

✅ Temporary payload prepared
{
  "bird": "Emperor penguin",
  "groupId": "Penguin",
  "group_is_new": true,
  "group_description": "Flightless aquatic birds known for their upright stance and specialized wings for swimming in the Southern Hemisphere.",
  "images": [
    "emperor_penguin_1.jpg",
    "emperor_penguin_2.jpg",
    "emperor_penguin_3.jpg",
    "emperor_penguin_4.jpg",
    "emperor_penguin_5.jpg"
  ],
  "payload_path": "C:\\Users\\matan\\Documents\\BirdWho_ag\\tmp\\new_bird_onboarding\\emperor_penguin\\new_bird_payload.json"
}


In [17]:
# Block 7: Integrate into app data (approval-gated)
# ONLY updates data/quiz_tiers_data.json (as requested).
# After successful apply, tmp onboarding folder is deleted.

CLEANUP_TMP_AFTER_APPLY = True

payload = load_json(TMP_ROOT / "new_bird_payload.json")
bird_slug = payload["birdSlug"]
species = payload["species"]
new_group = payload.get("group")

qt_path = DATA_DIR / "quiz_tiers_data.json"


def _norm_slug(v: str) -> str:
    v = str(v or "").strip().lower()
    v = re.sub(r"[^a-z0-9]+", "_", v)
    return re.sub(r"_+", "_", v).strip("_")


def _norm_desc(d):
    d = d if isinstance(d, dict) else {}
    return {
        "en": str(d.get("en", "")).strip()[:220],
        "he": str(d.get("he", "")).strip()[:220],
        "es": str(d.get("es", "")).strip()[:220],
        "fr": str(d.get("fr", "")).strip()[:220],
    }


def _normalize_species_entity(s: dict):
    group_id = _norm_slug(s.get("groupId", ""))
    sid = _norm_slug(s.get("id", "")) or _norm_slug(s.get("name", ""))
    return {
        "id": sid,
        "name": sid,
        "names": s.get("names", {}),
        "scientificName": str(s.get("scientificName", "Unknown")),
        "difficulty": int(s.get("difficulty", 3)),
        "tier": 3,
        "entryType": "species",
        "category": s.get("category", "Songbird"),
        "locations": s.get("locations", []) if isinstance(s.get("locations", []), list) else [],
        "images": s.get("images", []) if isinstance(s.get("images", []), list) else [],
        "image": sid,
        "groupId": group_id,
        "isGroup": False,
        "description": _norm_desc(s.get("description", {})),
    }


def _normalize_group_entity(g: dict, fallback_image: str):
    gid = _norm_slug(g.get("groupId", "") or g.get("name", ""))
    return {
        "id": f"group_{gid}",
        "name": gid,
        "names": g.get("names", {}),
        "scientificName": "Various",
        "difficulty": int(g.get("difficulty", 2)),
        "tier": int(g.get("tier", 2)) if int(g.get("tier", 2)) in [1, 2] else 2,
        "entryType": "group",
        "category": g.get("category", "Songbird"),
        "locations": g.get("locations", []) if isinstance(g.get("locations", []), list) else [],
        "tags": g.get("tags", []) if isinstance(g.get("tags", []), list) else [],
        "images": g.get("images", []) if isinstance(g.get("images", []), list) else [],
        "image": g.get("image", fallback_image),
        "groupId": gid,
        "isGroup": True,
        "description": _norm_desc(g.get("description", {})),
    }


if not APPLY_CHANGES:
    print("🛑 APPLY_CHANGES=False -> preview only, no files changed.")
    print(json.dumps({
        "would_add_species": bird_slug,
        "would_add_group": (new_group or {}).get("id"),
        "target_primary": str(qt_path),
        "cleanup_tmp_after_apply": CLEANUP_TMP_AFTER_APPLY,
        "tmp_folder": str(TMP_ROOT),
    }, ensure_ascii=False, indent=2))
else:
    # keep backup outside onboarding tmp so cleanup won't remove it
    backup_dir = REPO_ROOT / "tmp" / "new_bird_backups" / f"{bird_slug}_{int(time.time())}"
    backup_dir.mkdir(parents=True, exist_ok=True)
    if qt_path.exists():
        shutil.copy2(qt_path, backup_dir / qt_path.name)

    # integrate into quiz_tiers_data.json
    qt = load_json(qt_path)
    qt.setdefault("tiers", {}).setdefault("1", [])
    qt.setdefault("tiers", {}).setdefault("2", [])
    qt.setdefault("tiers", {}).setdefault("3", [])
    qt.setdefault("entities", {})

    species_norm = _normalize_species_entity(species)
    species_id = species_norm["id"]

    qt["entities"][species_id] = species_norm
    for t in ["1", "2"]:
        if species_id in qt["tiers"][t]:
            qt["tiers"][t].remove(species_id)
    if species_id not in qt["tiers"]["3"]:
        qt["tiers"]["3"].append(species_id)

    group_added_id = None
    if new_group:
        group_norm = _normalize_group_entity(new_group, fallback_image=species_id)
        group_key = group_norm["id"]
        group_added_id = group_key

        qt["entities"][group_key] = group_norm
        for t in ["1", "2"]:
            if group_key in qt["tiers"][t]:
                qt["tiers"][t].remove(group_key)
        target_group_tier = str(group_norm["tier"])
        if group_key not in qt["tiers"][target_group_tier]:
            qt["tiers"][target_group_tier].append(group_key)

        qt["entities"][species_id]["groupId"] = group_norm["groupId"]

    save_json(qt_path, qt)

    summary = {
        "status": "applied",
        "primary_file_updated": str(qt_path),
        "species_added": species_id,
        "group_added": group_added_id,
        "group_used": qt["entities"][species_id].get("groupId"),
        "backup_dir": str(backup_dir),
    }
    save_json(TMP_ROOT / "integration_summary.json", summary)
    print("✅ Integration completed")
    print(json.dumps(summary, ensure_ascii=False, indent=2))

    if CLEANUP_TMP_AFTER_APPLY and TMP_ROOT.exists():
        shutil.rmtree(TMP_ROOT, ignore_errors=True)
        print(f"🧹 Deleted tmp onboarding folder: {TMP_ROOT}")

✅ Integration completed
{
  "status": "applied",
  "primary_file_updated": "C:\\Users\\matan\\Documents\\BirdWho_ag\\data\\quiz_tiers_data.json",
  "species_added": "emperor_penguin",
  "group_added": "group_penguin",
  "group_used": "penguin",
  "backup_dir": "C:\\Users\\matan\\Documents\\BirdWho_ag\\tmp\\new_bird_backups\\emperor_penguin_1773849062"
}
🧹 Deleted tmp onboarding folder: C:\Users\matan\Documents\BirdWho_ag\tmp\new_bird_onboarding\emperor_penguin
